# DS4DS Exercise Sheet 02

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

In [ ]:
using Plots
using HDF5

### Task 1: Solving the mathematical pendulum with different numerical ODE solvers
Hint: All subtasks can be done using Julia

Consider the mathematical pendulum (without friction) depicted in the image below. Its motion is described by the ODE

$$
\begin{align}
    \frac{\mathrm{d}^2\theta(t)}{\mathrm{d}t^2} + \frac{g}{l} \sin{\theta(t)} = 0,
\end{align}
$$

where $g = 9.81 \frac{\mathrm{m}}{\mathrm{s}^2}$ is the magnitude of the gravitational field, $l = 1 \mathrm{m}$ is the length of the rod and $\theta$ is the angle from the vertical of the pendulum.
To transfer this ODE into the state-space representation, we choose the state vector 

$$
\begin{align}
\boldsymbol{x}(t) = \begin{bmatrix}x_1(t) \\ x_2(t) \end{bmatrix} = \begin{bmatrix}\theta(t) \\ \omega(t) \end{bmatrix} = 
            \begin{bmatrix} \theta(t) \\\frac{\mathrm{d}\theta(t)}{\mathrm{d}t} \end{bmatrix}.
\end{align}
$$

This results in the state-space representation
$$
\begin{align}
    \frac{\mathrm{d}\boldsymbol{x}(t)}{\mathrm{d}t} = \begin{bmatrix}\frac{\mathrm{d}\theta(t)}{\mathrm{d}t}\\ \frac{\mathrm{d}^2\theta(t)}{\mathrm{d}t^2} \end{bmatrix} =  \begin{bmatrix}\omega(t) \\ -\frac{g}{l} \sin{(\theta(t))} \end{bmatrix} = \begin{bmatrix}x_2(t) \\ -\frac{g}{l} \sin{(x_1(t))} \end{bmatrix}.
\end{align}
$$

The goal of this task is to compute the solution $\boldsymbol{x}(t)$ of this ODE numerically, when the intial state is given as $\boldsymbol{x}_0 = \begin{bmatrix} \frac{\pi}{2} & 0 \end{bmatrix}^\mathrm{T} $. This means we are looking for the values for the angle $\theta$ and the angular velocity $\omega$, when, initially, the pendulum is in a horizontal position and has no angular velocity and is then let go. To enable the usage of numerical solvers, we have to choose discrete points in time at which we compute these values. This means we only consider $\boldsymbol{x}[k] = \boldsymbol{x}(k \cdot \Delta t)$.

**Source ODE & description:** https://en.wikipedia.org/wiki/Pendulum_(mechanics)

![math pendulum](Mathematical_pendulum.png)

**Source Image:** By Subjektivisti - Own work, CC0, https://commons.wikimedia.org/w/index.php?curid=63812625


In [ ]:
# parameters
g = 9.81
l = 1

function pendulum_ode(x)
	"""Computes the derivative of the state given the current state."""
	dx = [x[2], -g / l * sin(x[1])]
	return dx
end

# initial state and system dimension
x0 = [pi / 2, 0]
n_states = 2;


(i) Implement functions for

- **a)** the explicit Euler method,
- **b)** the improved Euler method,
- **c)** the classical Runge-Kutta scheme of fourth order (RK4: https://en.wikipedia.org/wiki/Runge-Kutta_methods),


and use your implementations to numerically compute the solution for $\boldsymbol{x}[k]$ in the time window $t \in [t_0, t_e] = [0\,\mathrm{s}, 10\,\mathrm{s}]$ with a step size of $\Delta t = 0.2\,\mathrm{s}$.

In [ ]:
# Time settings
t0, te = 0, 10
dt = 0.2

N = Int((te - t0) / dt + 1)
time_values = LinRange(t0, te, N);

**a)** Implement the explicit Euler method

In [ ]:
function explicit_euler_solver(n_states, N, dt, x0, ode)
    """Uses the explicit Euler method to numerically solve the given ode.

    Args:
        n_states: The number of states in the given system.
        N: The number of discrete timesteps in the time window.
        dt: The length of each of the timesteps.
        x0: A vector with shape (n_states) containing the initial state of the system.
        ode: The ODE implementation. This is a function that takes the current state as input 
            and returns the derivative at the given state.
        
    Returns:
        The resulting state trajectory x[k] in an array with size (n_states, N).
    """
    ### BEGIN SOLUTION
    x = zeros(n_states, N)

    x[:, 1] = x0
    for k in 2:N
        x[:, k] = x[:, k-1] + dt * ode(x[:, k-1])
    end

    return x
    ### END SOLUTION
end;


**b)** Implement the improved Euler method

We have an inconsistency between the *improved Euler* definition from the lecture and the general literature. The lecture teaches the *midpoint method* and the literature defines it under an alternative name which is *Heun's method*. We allow both in this task.

Sidenote: The *midpoint method* is also called *Runge-Kutta method of second order with two stages and alpha = 1/2* and *Heun's method* is also called *Runge-Kutta method of second order with two stages and alpha = 1*.


In [ ]:
function improved_euler_solver(n_states, N, dt, x0, ode)
	"""Uses the improved euler method to numerically solve the given ode.

	Args:
		n_states: The number of states in the given system.
		N: The number of discrete timesteps in the time window.
		dt: The length of each of the timesteps.
		x0: A vector with shape (n_states) containing the initial state of the system.
		ode: The ODE implementation. This is a function that takes the current state as input 
			and returns the derivative at the given state.
		
	Returns:
		The resulting state trajectory x[k] in an array with size (n_states, N).
	"""
	### BEGIN SOLUTION
	x = zeros(n_states, N)

	x[:, 1] = x0
    
    # Explicit midpoint method (called "improved Euler" in the lecture or
    # "Runge-Kutta method of second order with two stages and alpha = 1/2" in the literature.)
	for k in 2:N
		x_half = x[:, k-1] + 0.5 * dt * ode(x[:, k-1])
		x[:, k] = x[:, k-1] + dt * ode(x_half)
	end
    
    # Heun's Method (also called "improved Euler" or
    # "Runge-Kutta method of second order with two stages and alpha = 1" in the literature )
#     for k in 2:N
#         k1 = ode(x[:, k-1])
#         k2 = ode(x[:, k-1] + dt * k1)
#         x[:, k] = x[:, k-1] + 0.5 * dt * (k1 + k2)
#     end

	return x
	### END SOLUTION
end;


**c)** Implement the classical Runge-Kutta scheme of fourth order (RK4: https://en.wikipedia.org/wiki/Runge-Kutta_methods)

In [ ]:
function RK4_solver(n_states, N, dt, x0, ode)
    """Uses the classical Runge-Kutta scheme of fourth order to numerically solve the given ode.

    Args:
        n_states: The number of states in the given system.
        N: The number of discrete timesteps in the time window.
        dt: The length of each of the timesteps.
        x0: A vector with shape (n_states) containing the initial state of the system.
        ode: The ODE implementation. This is a function that takes the current state as input 
            and returns the derivative at the given state.

    Returns:
        The resulting state trajectory x[k] in an array with size (n_states, N).
    """
    ### BEGIN SOLUTION
    x = zeros(n_states, N)

    x[:, 1] = x0
    for k in 2:N
        a1 = ode(x[:, k-1])
        a2 = ode(x[:, k-1] + 0.5 * dt * a1)
        a3 = ode(x[:, k-1] + 0.5 * dt * a2)
        a4 = ode(x[:, k-1] + dt * a3)

        x[:, k] = x[:, k-1] + 1 / 6 * dt * (a1 + 2 * a2 + 2 * a3 + a4)
    end

    return x
    ### END SOLUTION
end;


Compute $\boldsymbol{x}[k]$ in the time window $t \in [t_0, t_e] = [0\,\mathrm{s}, 10\,\mathrm{s}]$ with a step size of $\Delta t = 0.2\,\mathrm{s}$

In [ ]:
x_EE = explicit_euler_solver(n_states, N, dt, x0, pendulum_ode);
x_ImprovedE = improved_euler_solver(n_states, N, dt, x0, pendulum_ode);
x_RK4 = RK4_solver(n_states, N, dt, x0, pendulum_ode);

You can use the following plots of the sample solution to check your results qualitatively.

![EE_plot](EE_dt02.png)

![IE_plot](ImprovedE_dt02.png)

![RK4_plot](RK4_dt02.png)

![quiver_plot](quiver_plot_dt02.png)


(ii) Compare the mean distance between the trajectories $\boldsymbol{x}[k]$ generated by all three methods to the exact solution $\boldsymbol{z_1}[k]$:

$$
\begin{align}
    \mathrm{MD(\boldsymbol{z}, \boldsymbol{x})} &= \frac{1}{N} \sum_{k=1}^{N} \Vert 
    \boldsymbol{x}[k] - \boldsymbol{z}[k]\Vert_2 \\
    &= \frac{1}{N} \sum_{k=1}^{N} \sqrt{[(x_1[k] - z_1[k])^2 + (x_2[k] - z_2[k])^2 + ... + (x_{d_x}[k] - z_{d_x}[k])^2]},
\end{align}
$$

where $d_x$ is the number of states in the trajectory and $N$ is the number of time steps.

In [ ]:
z1_dt_200ms = h5read("z1_dt_200ms.h5", "z1_dt_200ms");

Implement a function for the calculation of the MD

In [ ]:
function MD(z, x)
    """Computes the mean distance between two input sequences with shapes (n_states, N)

    Args:
        z: First input matrix
        x: Second input matrix

    Returns:
        MD_value: A single float value that indicates the mean distance between the two inputs.
    """
    ### BEGIN SOLUTION
        
    distances = sqrt.(sum((z - x).^2, dims=1))
    if length(size(z)) > 1
        MD_value = 1 / size(z)[2] * sum(distances)
    else
        MD_value = sum(distances)
    end

    ### END SOLUTION
    return MD_value
end


Use your implementation of the MD to compute the distance between $\mathbf{z_1}[k]$ and your state-trajectories form task (i)

In [ ]:
### BEGIN SOLUTION

MD_EE = MD(z1_dt_200ms, x_EE);
MD_ImprovedE = MD(z1_dt_200ms, x_ImprovedE);
MD_RK4 = MD(z1_dt_200ms, x_RK4);

### END SOLUTION

(iii) Reduce the step size by a factor of two. Compute the ratio between the new MD and the MD from (ii) for all three solvers:

$$
\begin{align}
    r = \frac{\mathrm{MD}_{(0.5 \cdot \Delta t)}}{\mathrm{MD}_{(\Delta t)}} .
\end{align}   
$$

**Hint:** You need to compare the $\boldsymbol{x}(t)$ simulated with $\Delta t=0.1\mathrm{s}$ with the $\boldsymbol{z}(t)$ with $\Delta t=0.1\mathrm{s}$ and analogously for $\Delta t=0.2\mathrm{s}$ to compute the respective MD values.

In [ ]:
z1_dt_100ms = h5read("z1_dt_100ms.h5", "z1_dt_100ms");

**a)** For the Explicit Euler 

In [ ]:
### BEGIN SOLUTION
x_EE_dt_100ms = explicit_euler_solver(n_states, (N - 1) * 2 + 1, dt * 0.5, x0, pendulum_ode)
r_EE = MD(z1_dt_100ms, x_EE_dt_100ms) / MD(z1_dt_200ms, x_EE);
### END SOLUTION


**b)** For the Improved Euler

In [ ]:
### BEGIN SOLUTION
x_ImprovedE_dt_100ms = improved_euler_solver(n_states, (N - 1) * 2 + 1, dt * 0.5, x0, pendulum_ode)
r_ImprovedE = MD(z1_dt_100ms, x_ImprovedE_dt_100ms) / MD(z1_dt_200ms, x_ImprovedE);
### END SOLUTION


**c)** For the classical Runge-Kutta scheme of fourth order

In [ ]:
### BEGIN SOLUTION
x_RK4_dt_100ms = RK4_solver(n_states, (N - 1) * 2 + 1, dt * 0.5, x0, pendulum_ode)
r_RK4 = MD(z1_dt_100ms, x_RK4_dt_100ms) / MD(z1_dt_200ms, x_RK4)
### END SOLUTION

**To verify your results:** Redo the plots from (i). Do the new state trajectories make sense to you, compared to the ones with longer timesteps? What do you observe / expect regarding the found values of $r$ for the different solver schemes?

### Task 2: Solving the Lorenz System

The Lorenz attractor is a mathematical model developed by Edward Lorenz in the 1960s. It describes chaotic behavior in a simplified weather system. The attractor's butterfly-shaped trajectory symbolizes chaos theory's key features, like sensitivity to initial conditions and unpredictability. It has wide applications in various fields beyond meteorology, offering insights into complex, unpredictable systems ([Further information](https://en.wikipedia.org/wiki/Lorenz_system)).

![Lorenz_system](Lorenz_system.png)

**Source Image:** By Wikimol - Own work, CC0, [Source](https://commons.wikimedia.org/wiki/File:Lorenz_system_r28_s10_b2-6666.png?uselang=en#Licensing)

The system is described by a system of 3 differential equations:

$$
\begin{align}
\frac{\mathrm{d}x_1}{\mathrm{d}t} &= \sigma (x_2 - x_1), \\
\frac{\mathrm{d}x_2}{\mathrm{d}t} &= x_1 \cdot (\rho - x_3) - x_2, \\
\frac{\mathrm{d}x_3}{\mathrm{d}t} &= x_1 \cdot x_2 - \beta x_3. \\
\end{align}
$$

The typical values for the Lorenz system are $\sigma = 10$, $\rho = 28$ and $\beta = 8/3$.

We will use the Lorenz attractor here to take a closer look at the solvers implemented in task 1 and investigate the impact of the stepsize.

In [ ]:
# num states
n_states = 3

t0, te = 0, 0.1
dt = 1e-5

# initial value
x0 = [1.0, 0.0, 0.0]

# parameters
params = [10, 28, 8 / 3]

function lorenz_ode(x)
	"""Computes the derivative of the state given the current state."""
	dx = zeros(3)
	dx[1] = params[1] * (x[2] - x[1])
	dx[2] = x[1] * (params[2] - x[3]) - x[2]
	dx[3] = x[1] * x[2] - params[3] * x[3]

	return dx
end;


Set the stepsize to a multiple of $10^{-5}$, i.e., $\Delta t = n \cdot 10^{-5}$. Simulate from $t_0=0$ to $t_e=0.1$ for the Runge-Kutta 4 from Task 1 and find the largest integer factor $n$ (i.e., the largest step size $\Delta t$) such that the error (measured by the 2-norm) between the simulated result at the end of the simulation $x(t_e)$ and the exact solution $z(t_e)$ at that time is less than $\epsilon = 10^{-3}$.

In mathematical formulation

$$
\begin{align}
\max \quad & n\\
\textrm{s.t.} \quad & \Vert \boldsymbol{x}(t_e) – \boldsymbol{z}(t_e)\Vert _2 \leq \epsilon .
\end{align}
$$

Note that the number of steps $N$ changes when the step size is adjusted. Therefore, the target array must also be adjusted for each new step size. If the number of steps is not an integer please use the `round` function to round it.


**Hint:** n is in the range $[1, 30]$. Use a for-loop to increment $n$ step by step and determine the distance to the correct end result in each iteration.

In [ ]:
# exact solution z(t_e)
z2_te = [
	1.271459653756519,
	2.550243695182362,
	0.11413846073908919,
]

In [ ]:
epsilon = 1e-3
dt = 1e-5

n_range = 1:30

n_solution = NaN

distance_te_RK4 = NaN

for n in n_range
	# calc new stepsize
	# calc new x_RK4(t_e)
	# check if distance > eps
	# set n_solution = n

	### BEGIN SOLUTION
	dt_ = dt * n
	N_ = Int(round((te - t0) / dt_) + 1)

	x_RK4 = RK4_solver(n_states, N_, dt_, x0, lorenz_ode)
    distance_te_RK4 = MD(z2_te, x_RK4[:, end])

    if distance_te_RK4 >= epsilon
		println(n - 1)
		n_solution = n - 1
		break
	end
	println(distance_te_RK4)
	### END SOLUTION
end
